# AI Sales Lead Prediction System
**Binary Classification | Target: `Converted` (1 = Converted, 0 = Not Converted)**

---

## Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, roc_curve)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

import os
os.makedirs('outputs', exist_ok=True)

print('All libraries imported successfully.')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')
import sklearn; print(f'Scikit-learn: {sklearn.__version__}')
print(f'XGBoost available: {XGBOOST_AVAILABLE}')

---
## STEP 1: DATA UNDERSTANDING

In [ ]:
# Load dataset
df = pd.read_csv('sales_leads.csv')
print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

In [ ]:
# First 5 rows
print('First 5 rows:')
df.head()

In [ ]:
# Dataset info
print('Dataset Info:')
df.info()

In [ ]:
# Summary statistics
print('Numerical Summary Statistics:')
df.describe()

In [ ]:
# Column names
print('All Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

---
## STEP 2: EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f'Total columns with missing values: {len(missing_df)}')
print()
print(missing_df.to_string())

In [ ]:
# Target variable distribution
target_counts = df['Converted'].value_counts()
target_pct = df['Converted'].value_counts(normalize=True) * 100

print('Target Variable Distribution:')
print(f'  Converted (1): {target_counts[1]} ({target_pct[1]:.2f}%)')
print(f'  Not Converted (0): {target_counts[0]} ({target_pct[0]:.2f}%)')
print(f'  Imbalance ratio: {target_counts[0]/target_counts[1]:.2f}:1')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Not Converted (0)', 'Converted (1)'], target_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.8)
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=['Not Converted', 'Converted'],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
            startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Conversion Rate', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to outputs/target_distribution.png')

In [ ]:
# Numerical feature distributions
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'Converted']
print(f'Numerical features: {num_cols}')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols[:6]):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# Correlation heatmap (numerical features)
num_df = df[num_cols + ['Converted']].copy()
corr_matrix = num_df.corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=mask, linewidths=0.5,
            annot_kws={'size': 10})
plt.title('Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlation with target (Converted):')
print(corr_matrix['Converted'].drop('Converted').sort_values(ascending=False))

In [ ]:
# Lead Source vs Conversion
if 'Lead Source' in df.columns:
    lead_conv = df.groupby('Lead Source')['Converted'].agg(['sum', 'count'])
    lead_conv['Conversion Rate %'] = (lead_conv['sum'] / lead_conv['count'] * 100).round(2)
    lead_conv = lead_conv.sort_values('Conversion Rate %', ascending=False)
    lead_conv.columns = ['Converted', 'Total', 'Conversion Rate %']

    print('Lead Source Conversion Rates:')
    print(lead_conv.to_string())

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(lead_conv.index, lead_conv['Conversion Rate %'],
                  color='steelblue', edgecolor='black', alpha=0.8)
    ax.set_title('Conversion Rate by Lead Source', fontsize=13, fontweight='bold')
    ax.set_ylabel('Conversion Rate (%)')
    ax.set_xlabel('Lead Source')
    plt.xticks(rotation=45, ha='right')
    for bar, val in zip(bars, lead_conv['Conversion Rate %']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig('outputs/lead_source_conversion.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Total Time on Website vs Conversion (boxplot)
if 'Total Time Spent on Website' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    df.boxplot(column='Total Time Spent on Website', by='Converted', ax=axes[0])
    axes[0].set_title('Time on Website vs Conversion')
    axes[0].set_xlabel('Converted (0=No, 1=Yes)')
    axes[0].set_ylabel('Total Time Spent (sec)')
    plt.suptitle('')

    df.boxplot(column='TotalVisits', by='Converted', ax=axes[1])
    axes[1].set_title('Total Visits vs Conversion')
    axes[1].set_xlabel('Converted (0=No, 1=Yes)')
    axes[1].set_ylabel('Total Visits')
    plt.suptitle('')

    plt.tight_layout()
    plt.savefig('outputs/website_engagement.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Website Engagement vs Conversion saved.')

---
## STEP 3: DATA PREPARATION

In [ ]:
# Drop identifier columns
id_cols = ['Prospect ID', 'Lead Number']
existing_id = [c for c in id_cols if c in df.columns]
df_clean = df.drop(columns=existing_id)
print(f'Dropped identifier columns: {existing_id}')
print(f'Shape after dropping identifiers: {df_clean.shape}')

# Separate features and target
X = df_clean.drop(columns=['Converted'])
y = df_clean['Converted']
print(f'Features (X) shape: {X.shape}')
print(f'Target (y) shape: {y.shape}')
print(f'Target distribution:\n{y.value_counts()}')

In [ ]:
# Missing value strategy
missing_pct = X.isnull().mean() * 100

high_missing  = missing_pct[missing_pct > 40].index.tolist()   # drop
med_missing   = missing_pct[(missing_pct > 10) & (missing_pct <= 40)].index.tolist()  # impute
low_missing   = missing_pct[(missing_pct > 0) & (missing_pct <= 10)].index.tolist()   # keep

print(f'Columns with >40% missing (DROP): {len(high_missing)}')
for c in high_missing: print(f'   - {c}: {missing_pct[c]:.1f}%')

print(f'\nColumns with 10-40% missing (IMPUTE): {len(med_missing)}')
for c in med_missing: print(f'   - {c}: {missing_pct[c]:.1f}%')

print(f'\nColumns with <10% missing (KEEP+IMPUTE): {len(low_missing)}')
for c in low_missing: print(f'   - {c}: {missing_pct[c]:.1f}%')

---
## STEP 4: DATA CLEANING & PREPROCESSING

In [ ]:
# Drop high missing columns
X = X.drop(columns=high_missing)
print(f'Shape after dropping high-missing columns: {X.shape}')
print(f'Remaining columns: {X.shape[1]}')

In [ ]:
# Identify numerical and categorical features
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical features ({len(num_features)}): {num_features}')
print()
print(f'Categorical features ({len(cat_features)}): {cat_features}')

In [ ]:
# Build preprocessing pipelines
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, num_features),
    ('cat', categorical_pipeline, cat_features)
])

print('Preprocessing pipelines built:')
print('  Numerical Pipeline: Median Imputation -> Standard Scaling')
print('  Categorical Pipeline: Most Frequent Imputation -> One-Hot Encoding')
print('  ColumnTransformer: applies pipelines to respective feature groups')

In [ ]:
# Fit and transform
X_processed = preprocessor.fit_transform(X)

# Get output column names
cat_encoder = preprocessor.named_transformers_['cat']['encoder']
cat_col_names = cat_encoder.get_feature_names_out(cat_features).tolist()
all_col_names = num_features + cat_col_names

X_processed_df = pd.DataFrame(X_processed, columns=all_col_names)

print('Preprocessing complete!')
print(f'Final processed shape: {X_processed_df.shape}')
print(f'Missing values remaining: {X_processed_df.isnull().sum().sum()}')
print(f'All dtypes numeric: {all(X_processed_df.dtypes != object)}')
print()
print('Sample of processed data (first 3 rows, first 10 cols):')
X_processed_df.iloc[:3, :10]

In [ ]:
# Validation summary
print('=== PREPROCESSING VALIDATION SUMMARY ===')
print(f'Final Shape      : {X_processed_df.shape}')
print(f'Missing Values   : {X_processed_df.isnull().sum().sum()}')
print(f'All Types Numeric: {all(X_processed_df.dtypes != object)}')
print(f'Numeric features : {len(num_features)}')
print(f'Categorical cols : {len(cat_features)} (expanded to {len(cat_col_names)} after OHE)')
print(f'Total features   : {len(all_col_names)}')

---
## STEP 5: STRATIFIED TRAIN-TEST SPLIT

In [ ]:
# Stratified 80/20 split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in sss.split(X_processed_df, y):
    X_train = X_processed_df.iloc[train_idx]
    X_test  = X_processed_df.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

print('Stratified Train-Test Split:')
print(f'  Train set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X_processed_df)*100:.1f}%)')
print(f'  Test set  : {X_test.shape[0]} samples ({X_test.shape[0]/len(X_processed_df)*100:.1f}%)')
print()
print('Class distribution preserved:')
print(f'  Train - Converted: {y_train.mean()*100:.2f}%')
print(f'  Test  - Converted: {y_test.mean()*100:.2f}%')
print(f'  Original          : {y.mean()*100:.2f}%')

---
## STEP 6: MODEL TRAINING

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBClassifier(n_estimators=100, use_label_encoder=False,
                                       eval_metric='logloss', random_state=42)

print(f'Models to train: {list(models.keys())}')

# Train all models
results = {}
for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model'    : model,
        'y_pred'   : y_pred,
        'y_prob'   : y_prob,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1 Score' : f1_score(y_test, y_pred),
        'ROC-AUC'  : roc_auc_score(y_test, y_prob),
    }
    print(f'Done. Accuracy={results[name]["Accuracy"]:.4f}')

print('\nAll models trained successfully!')

---
## STEP 7: MODEL EVALUATION

In [ ]:
# Metrics comparison table
metrics_df = pd.DataFrame({
    name: {
        'Accuracy' : r['Accuracy'],
        'Precision': r['Precision'],
        'Recall'   : r['Recall'],
        'F1 Score' : r['F1 Score'],
        'ROC-AUC'  : r['ROC-AUC'],
    }
    for name, r in results.items()
}).T.round(4)

print('Model Performance Comparison:')
print(metrics_df.to_string())

In [ ]:
# Metrics bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_df))
width = 0.15
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, (metric, color) in enumerate(zip(metric_cols, colors)):
    bars = ax.bar(x + i * width, metrics_df[metric], width, label=metric,
                  color=color, alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_df.index, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='0.8 baseline')

plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Model comparison plot saved.')

In [ ]:
# ROC curves for all models
plt.figure(figsize=(9, 7))

colors_roc = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={r['ROC-AUC']:.4f})",
             color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('ROC curves saved.')

In [ ]:
# Confusion matrices for all models
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Conv.', 'Converted'],
                yticklabels=['Not Conv.', 'Converted'],
                linewidths=1)
    ax.set_title(f'{name}\nF1={r["F1 Score"]:.4f}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrices saved.')

In [ ]:
# Detailed classification reports
for name, r in results.items():
    print(f'=== {name} ===')
    print(classification_report(y_test, r['y_pred'],
                                 target_names=['Not Converted', 'Converted']))
    print()

---
## STEP 8: MODEL SELECTION

In [ ]:
# Select best model by ROC-AUC
best_model_name = metrics_df['ROC-AUC'].idxmax()
best_model = results[best_model_name]['model']

print('=== MODEL SELECTION RESULT ===')
print(f'Best Model: {best_model_name}')
print()
print('Performance Metrics:')
for metric in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']:
    val = results[best_model_name][metric]
    print(f'  {metric:<12}: {val:.4f}')

print()
print('Ranked Models by ROC-AUC:')
ranked = metrics_df.sort_values('ROC-AUC', ascending=False)
for i, (name, row) in enumerate(ranked.iterrows(), 1):
    marker = ' <-- BEST' if name == best_model_name else ''
    print(f'  {i}. {name:<22} ROC-AUC={row["ROC-AUC"]:.4f}{marker}')

In [ ]:
# Feature importance (for tree-based best model)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=all_col_names).sort_values(ascending=False)

    print(f'Top 15 Important Features ({best_model_name}):')
    print(feat_imp.head(15).to_string())

    plt.figure(figsize=(10, 6))
    feat_imp.head(15).plot(kind='barh', color='steelblue', edgecolor='black', alpha=0.8)
    plt.xlabel('Feature Importance')
    plt.title(f'Top 15 Feature Importances ({best_model_name})', fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Feature importance plot saved.')
elif hasattr(best_model, 'coef_'):
    coef = pd.Series(np.abs(best_model.coef_[0]), index=all_col_names).sort_values(ascending=False)
    print(f'Top 15 Feature Coefficients (Logistic Regression):')
    print(coef.head(15).to_string())

    plt.figure(figsize=(10, 6))
    coef.head(15).plot(kind='barh', color='steelblue', edgecolor='black', alpha=0.8)
    plt.xlabel('|Coefficient|')
    plt.title(f'Top 15 Feature Coefficients (Logistic Regression)', fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Final summary
print('===================================================')
print('        AI SALES LEAD PREDICTION SYSTEM           ')
print('             FINAL PROJECT SUMMARY                ')
print('===================================================')
print(f'Dataset        : sales_leads.csv')
print(f'Original Shape : {df.shape}')
print(f'Processed Shape: {X_processed_df.shape}')
print(f'Target Variable: Converted (Binary)')
print(f'Train/Test     : 80% / 20% (Stratified)')
print(f'Train Samples  : {X_train.shape[0]}')
print(f'Test Samples   : {X_test.shape[0]}')
print()
print(f'BEST MODEL     : {best_model_name}')
for metric in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']:
    print(f'  {metric:<12}: {results[best_model_name][metric]:.4f}')
print()
print('Output plots saved in: outputs/')
import os
for f in sorted(os.listdir('outputs')):
    print(f'  - {f}')
print('===================================================')